# Luồng nạp 1 frame vào gru
Các bước: 
- Nạp checkpoint
- Lấy 1 frame
- Gọi YOLO
- Đưa frame vào GRU

In [14]:
import sys
from pathlib import Path

import cv2
from ultralytics import YOLO

ROOT = Path.cwd()
if not (ROOT / "Hellu.py").is_file():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.detectors.gru import SequenceFallDetector, load_checkpoint
from src.utils.keypoints import person_keypoints

ROOT

WindowsPath('d:/Project/GitPJ/FD-Yolo/FD-Yolo')

## Nạp checkpoint
`model` là GRU đã train. `config` là các số đã lưu cùng trọng số, trong đó có `min_frames`

In [15]:
model, config = load_checkpoint(ROOT / "runs" / "fall-gru" / "checkpoint.pt")
config

{'target_fps': 10.0,
 'num_keypoints': 17,
 'hidden_size': 64,
 'num_layers': 1,
 'min_frames': 10,
 'decision_threshold': 0.5,
 'pose_model': 'yolo11x-pose.pt',
 'feature_order': 'xy_conf'}

## Đọc 1 frame từ falldemo.mp4
`frame.shape` là `(chiều cao, chiều rộng, 3)` 
3 là 3màu

In [16]:
video = ROOT / "banana-peel-fall.mp4"
capture = cv2.VideoCapture(str(video))
ok, frame = capture.read()
capture.release()
if not ok:
    raise RuntimeError(f"Không đọc được khung đầu của {video}")
frame.shape

(480, 852, 3)

## Gọi YOLO và in keypoints.shape
Ô này dùng `predict` vì chỉ có một khung, chưa cần `track_id`. `Hellu.py` dùng `track` khi chạy camera

`conf` là điểm của hộp người. Train và Hellu.py cùng dùng 0.25. Hộp dưới ngưỡng này bị bỏ.

In [17]:
conf = 0.25
pose = YOLO(str(ROOT / config["pose_model"]))
result = pose.predict(source=frame, verbose=False, conf=conf)[0]
keypoints = person_keypoints(result, int(config["num_keypoints"]))
print(keypoints.shape)
keypoints[:2]

(17, 3)


array([[    0.51228,     0.18691,     0.97058],
       [    0.51085,     0.17561,     0.26178]], dtype=float32)

## Đưa một khung vào GRU
`update` trả `None` khi chưa đủ `min_frames` khung. Checkpoint hiện tại chờ 10 khung, tức khoảng một giây

In [18]:
detector = SequenceFallDetector(
    model,
    target_fps=float(config["target_fps"]),
    min_frames=int(config["min_frames"]),
)
score = detector.update(keypoints, timestamp=0.0)
print(score)
print(detector.accepted_frames, "/", detector.min_frames)

None
1 / 10


## Đủ số khung rồi in xác suất

Cùng mảng khớp của khung đầu, gửi lại 10 lần. Mỗi lần cách 1 giây, dài hơn nhịp 0.1 giây

In [ ]:
detector.reset()
score = None
for index in range(int(config["min_frames"])):
    score = detector.update(keypoints, timestamp=float(index))
print(score)
print(detector.accepted_frames, "/", detector.min_frames)